# EDA And Dashboard-Ready Tables

This notebook creates the exploratory analysis layer for the Medicaid Enrollment & Eligibility Operations Dashboard project. It uses the validated processed CMS/Data.Medicaid.gov files and creates dashboard-ready tables for a future Plotly/Dash app.

This is not the dashboard build step. No forecasting is added here.

## Documentation Standard

Each major section answers:

- What did I do?
- Why did I do it?
- What does it mean for healthcare reporting, Medicaid policy monitoring, or operational decision-making?
- What are the data limitations?

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from build_dashboard_tables import save_dashboard_tables, save_figures
from eda_medicaid import (
    CLEAN_DATA_PATH,
    dashboard_notes,
    data_quality_by_field,
    data_quality_by_month,
    data_quality_by_state,
    load_clean_data,
    national_applications_determinations_trend,
    national_enrollment_trend,
    state_eligibility_operations_summary,
    state_enrollment_change,
    state_latest_snapshot,
    top_state_decreases,
    top_state_increases,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
print(PROJECT_ROOT)

/Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard


## 1. Load And Inspect Cleaned Data

What did I do? Loaded the cleaned state-month Medicaid/CHIP dataset and confirmed row count, date range, state/DC coverage, available fields, and uniqueness.

Why did I do it? The dashboard should be built only after confirming the data has the expected grain and coverage.

What does it mean? A one-row-per-state-month panel can support trend analysis, state comparison, and dashboard-ready summary tables.

Limitations: The latest month may be preliminary, and some operational fields have high missingness.

In [2]:
df = load_clean_data()
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Date range: {df['reporting_month'].min().date()} to {df['reporting_month'].max().date()}")
print(f"States/DC: {df['state_abbreviation'].nunique()}")
print(f"Duplicate state-month records: {df.duplicated(['state_abbreviation', 'reporting_month']).sum():,}")
latest_month = df['reporting_month'].max()
latest_prelim = int((df.loc[df['reporting_month'] == latest_month, 'final_report'] != 'Y').sum())
print(f"Latest reporting month: {latest_month.date()}")
print(f"Latest-month preliminary records: {latest_prelim}")
df.head()

Rows: 4,386
Columns: 27
Date range: 2019-01-01 to 2026-02-01
States/DC: 51
Duplicate state-month records: 0
Latest reporting month: 2026-02-01
Latest-month preliminary records: 51


,state_abbreviation,state_name,reporting_period,reporting_month,reporting_year,state_expanded_medicaid,preliminary_or_updated,final_report,new_applications_submitted_to_medicaid_and_chip_agencies,applications_for_financial_assistance_submitted_to_the_state_based_marketplace,total_applications_for_financial_assistance_submitted_at_state_level,individuals_determined_eligible_for_medicaid_at_application,individuals_determined_eligible_for_chip_at_application,total_medicaid_and_chip_determinations,medicaid_and_chip_child_enrollment,total_medicaid_and_chip_enrollment,total_medicaid_enrollment,total_chip_enrollment,total_adult_medicaid_enrollment,total_medicaid_and_chip_determinations_processed_in_less_than_24_hours,total_medicaid_and_chip_determinations_processed_between_24_hours_and_7_days,total_medicaid_and_chip_determinations_processed_between_8_days_and_30_days,total_medicaid_and_chip_determinations_processed_between_31_days_and_45_days,total_medicaid_and_chip_determinations_processed_in_more_than_45_days,total_call_center_volume_number_of_calls,average_call_center_wait_time_minutes,average_call_center_abandonment_rate
0,AK,Alaska,201901,2019-01-01,2019,Y,U,Y,3032.0,0.0,3032.0,3156.0,158.0,3314.0,95796.0,216175.0,201038.0,15137.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,Alaska,201902,2019-02-01,2019,Y,U,Y,3865.0,0.0,3865.0,3084.0,173.0,3257.0,96172.0,217468.0,202090.0,15378.0,NaN,492.0,833.0,699.0,281.0,3613.0,NaN,NaN,NaN
2,AK,Alaska,201903,2019-03-01,2019,Y,U,Y,3588.0,0.0,3588.0,3363.0,271.0,3634.0,96256.0,217793.0,202397.0,15396.0,NaN,586.0,835.0,878.0,332.0,5572.0,NaN,NaN,NaN
3,AK,Alaska,201904,2019-04-01,2019,Y,U,Y,3368.0,0.0,3368.0,3053.0,202.0,3255.0,96593.0,218972.0,203524.0,15448.0,NaN,535.0,818.0,1064.0,343.0,3829.0,NaN,NaN,NaN
4,AK,Alaska,201905,2019-05-01,2019,Y,U,Y,3107.0,0.0,3107.0,2675.0,113.0,2788.0,97003.0,220384.0,204803.0,15581.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print("Available fields:")
for column in df.columns:
    print(f"- {column}")

Available fields:
- state_abbreviation
- state_name
- reporting_period
- reporting_month
- reporting_year
- state_expanded_medicaid
- preliminary_or_updated
- final_report
- new_applications_submitted_to_medicaid_and_chip_agencies
- applications_for_financial_assistance_submitted_to_the_state_based_marketplace
- total_applications_for_financial_assistance_submitted_at_state_level
- individuals_determined_eligible_for_medicaid_at_application
- individuals_determined_eligible_for_chip_at_application
- total_medicaid_and_chip_determinations
- medicaid_and_chip_child_enrollment
- total_medicaid_and_chip_enrollment
- total_medicaid_enrollment
- total_chip_enrollment
- total_adult_medicaid_enrollment
- total_medicaid_and_chip_determinations_processed_in_less_than_24_hours
- total_medicaid_and_chip_determinations_processed_between_24_hours_and_7_days
- total_medicaid_and_chip_determinations_processed_between_8_days_and_30_days
- total_medicaid_and_chip_determinations_processed_between_31_days

## 2. National Enrollment Trend Analysis

What did I do? Created national monthly totals for Medicaid/CHIP enrollment, Medicaid enrollment, CHIP enrollment, monthly change, percent monthly change, and year-over-year change.

Why did I do it? National trends provide the overview layer for the future dashboard and help identify broad periods of increase or decrease.

What does it mean? These are descriptive aggregate enrollment trends, useful for monitoring and policy discussion.

Limitations: The trends do not identify causal policy effects. The latest month is preliminary and should be labeled cautiously.

In [4]:
national = national_enrollment_trend(df)
peak = national.loc[national['total_medicaid_and_chip_enrollment'].idxmax()]
baseline = national.iloc[0]
latest = national.iloc[-1]
print(f"Baseline enrollment ({baseline['reporting_month']:%Y-%m}): {baseline['total_medicaid_and_chip_enrollment']:,.0f}")
print(f"Peak enrollment ({peak['reporting_month']:%Y-%m}): {peak['total_medicaid_and_chip_enrollment']:,.0f}")
print(f"Latest enrollment ({latest['reporting_month']:%Y-%m}): {latest['total_medicaid_and_chip_enrollment']:,.0f}")
print(f"Change since baseline: {latest['total_medicaid_and_chip_enrollment'] - baseline['total_medicaid_and_chip_enrollment']:,.0f}")
print(f"Latest monthly change: {latest['monthly_change']:,.0f}")
national.tail(15)

Baseline enrollment (2019-01): 72,165,928
Peak enrollment (2023-04): 94,663,019
Latest enrollment (2026-02): 74,877,742
Change since baseline: 2,711,814
Latest monthly change: -918,199


,reporting_month,total_medicaid_and_chip_enrollment,total_medicaid_enrollment,total_chip_enrollment,state_records,preliminary_records,monthly_change,percent_monthly_change,year_over_year_change,percent_year_over_year_change
71,2024-12-01,79162078.0,71835380.0,7326698.0,51,0,-435893.0,-0.547618,-6599403.0,-7.695067
72,2025-01-01,79393536.0,72010630.0,7382906.0,51,0,231458.0,0.292385,-5459918.0,-6.434527
73,2025-02-01,79296495.0,71896356.0,7400139.0,51,0,-97041.0,-0.122228,-4892528.0,-5.811361
74,2025-03-01,79127297.0,71743787.0,7383510.0,51,0,-169198.0,-0.213374,-4327206.0,-5.185108
75,2025-04-01,78875273.0,71540589.0,7334684.0,51,0,-252024.0,-0.318504,-3523435.0,-4.276080
76,2025-05-01,78551179.0,71228849.0,7322330.0,51,0,-324094.0,-0.410894,-2927942.0,-3.593487
77,2025-06-01,78223264.0,70929378.0,7293886.0,51,0,-327915.0,-0.417454,-2369065.0,-2.939566
78,2025-07-01,78028646.0,70747932.0,7280714.0,51,0,-194618.0,-0.248798,-2221951.0,-2.768766
79,2025-08-01,77736256.0,70460344.0,7275912.0,51,0,-292390.0,-0.374721,-2438221.0,-3.041144
80,2025-09-01,77486188.0,70199433.0,7286755.0,51,0,-250068.0,-0.321688,-2531566.0,-3.163755


Short interpretation: National Medicaid/CHIP enrollment was relatively flat in 2019, rose through the early pandemic-era period, peaked in April 2023, and then declined through the most recent months. The February 2026 value is preliminary for all states/DC, so it should be shown with a caution note rather than treated as final.

## 3. State Comparison Analysis

What did I do? Created state-level summaries for latest enrollment, change from January 2019 to the latest month, percent change, last-12-month change, and state ranks.

Why did I do it? State comparison is central for Medicaid policy monitoring because states administer Medicaid and vary in enrollment trends and operations.

What does it mean? These tables show which states stand out descriptively and where analysts may investigate further.

Limitations: State changes should not be interpreted as caused by a specific policy without additional causal design and context.

In [5]:
state_change = state_enrollment_change(df)
print("Top 10 increases since Jan 2019:")
display(top_state_increases(state_change, 10)[['state_abbreviation','state_name','change_since_baseline','percent_change_since_baseline','change_last_12_months']])
print("Top 10 decreases since Jan 2019:")
display(top_state_decreases(state_change, 10)[['state_abbreviation','state_name','change_since_baseline','percent_change_since_baseline','change_last_12_months']])

Top 10 increases since Jan 2019:


,state_abbreviation,state_name,change_since_baseline,percent_change_since_baseline,change_last_12_months
27,NC,North Carolina,937677.0,49.050304,-13916.0
4,CA,California,533529.0,4.478020,-1050743.0
45,VA,Virginia,423525.0,34.230826,-139098.0
24,MO,Missouri,334268.0,36.817873,-71397.0
37,OR,Oregon,305757.0,31.099541,-29473.0
34,NY,New York,284084.0,4.627538,-188622.0
36,OK,Oklahoma,238581.0,32.675167,-9566.0
15,IN,Indiana,98741.0,7.371251,-338973.0
48,WI,Wisconsin,97291.0,9.524784,-72804.0
33,NV,Nevada,92106.0,14.500452,-44786.0


Top 10 decreases since Jan 2019:


,state_abbreviation,state_name,change_since_baseline,percent_change_since_baseline,change_last_12_months
43,TX,Texas,-299932.0,-6.978922,-200982.0
18,LA,Louisiana,-184356.0,-11.880208,-158664.0
22,MI,Michigan,-140641.0,-6.022924,-207924.0
9,FL,Florida,-99743.0,-2.710415,-157281.0
5,CO,Colorado,-94348.0,-7.381300,-30415.0
40,SC,South Carolina,-81233.0,-7.748682,-96330.0
26,MT,Montana,-72304.0,-25.780963,-5767.0
3,AZ,Arizona,-62309.0,-3.662112,-220149.0
32,NM,New Mexico,-54478.0,-7.416160,-43493.0
2,AR,Arkansas,-51862.0,-6.098943,-25763.0


In [6]:
snapshot = state_latest_snapshot(df)
snapshot.head(10)

,state_abbreviation,state_name,reporting_month,preliminary_or_updated,final_report,total_medicaid_and_chip_enrollment,total_medicaid_enrollment,total_chip_enrollment,total_applications_for_financial_assistance_submitted_at_state_level,total_medicaid_and_chip_determinations,national_enrollment_rank
429,CA,California,2026-02-01,P,N,12447923.0,11206969.0,1240954.0,172960.0,150977.0,1.0
3009,NY,New York,2026-02-01,P,N,6423072.0,5765633.0,657439.0,588083.0,116976.0,2.0
3783,TX,Texas,2026-02-01,P,N,3997752.0,3651928.0,345824.0,98889.0,59289.0,3.0
859,FL,Florida,2026-02-01,P,N,3580247.0,3414969.0,165278.0,239516.0,143927.0,4.0
3353,PA,Pennsylvania,2026-02-01,P,N,2994399.0,2715845.0,278554.0,101139.0,85575.0,5.0
1289,IL,Illinois,2026-02-01,P,N,2988258.0,2666845.0,321413.0,133810.0,38802.0,6.0
2407,NC,North Carolina,2026-02-01,P,N,2849341.0,2503652.0,345689.0,30062.0,44892.0,7.0
3095,OH,Ohio,2026-02-01,P,N,2681863.0,2439802.0,242061.0,68183.0,35274.0,8.0
1977,MI,Michigan,2026-02-01,P,N,2194454.0,2027525.0,166929.0,47799.0,34362.0,9.0
945,GA,Georgia,2026-02-01,P,N,1841369.0,1659357.0,182012.0,57044.0,21715.0,10.0


Short interpretation: North Carolina, California, Virginia, Missouri, and Oregon show the largest enrollment increases since January 2019 in this cleaned panel. Texas, Louisiana, Michigan, Florida, and Colorado show the largest decreases. These are descriptive comparisons; a policy analyst should examine state eligibility rules, expansion status, renewal timing, demographic context, and reporting notes before interpreting why states changed.

## 4. Eligibility Operations Analysis

What did I do? Summarized applications submitted, Medicaid/CHIP determinations, Medicaid determinations, CHIP determinations, and descriptive ratios such as determinations per application and applications per 1,000 enrollees.

Why did I do it? Applications and determinations provide an operations-facing complement to enrollment trends.

What does it mean? These fields can support descriptive monitoring of application and eligibility workload patterns.

Limitations: These ratios are not perfect performance metrics. They do not measure timeliness, accuracy, applicant experience, pending workload, or renewal outcomes by themselves.

In [7]:
ops = national_applications_determinations_trend(df)
print("Latest national operations indicators:")
print(ops.tail(1).T.to_string())
ops.tail(12)

Latest national operations indicators:
                                                         85
reporting_month                         2026-02-01 00:00:00
applications_submitted                            2460804.0
medicaid_chip_agency_applications                 1420622.0
total_medicaid_and_chip_determinations            1601892.0
medicaid_determinations                           1457448.0
chip_determinations                                144444.0
total_medicaid_and_chip_enrollment               74877742.0
state_records                                            51
preliminary_records                                      51
determinations_per_application                     0.650963
applications_per_1000_enrollees                   32.864292
determinations_per_1000_enrollees                 21.393434


,reporting_month,applications_submitted,medicaid_chip_agency_applications,total_medicaid_and_chip_determinations,medicaid_determinations,chip_determinations,total_medicaid_and_chip_enrollment,state_records,preliminary_records,determinations_per_application,applications_per_1000_enrollees,determinations_per_1000_enrollees
74,2025-03-01,2488450.0,1556963.0,1851068.0,1690174.0,160894.0,79127297.0,51,0,0.743864,31.448692,23.393545
75,2025-04-01,2467877.0,1571597.0,1809103.0,1653972.0,155131.0,78875273.0,51,0,0.733060,31.288348,22.936250
76,2025-05-01,2305722.0,1465102.0,1715563.0,1568568.0,146995.0,78551179.0,51,0,0.744046,29.353118,21.840067
77,2025-06-01,2300699.0,1435157.0,1613749.0,1474049.0,139700.0,78223264.0,51,0,0.701417,29.411953,20.630039
78,2025-07-01,2444547.0,1561189.0,1679879.0,1534875.0,145004.0,78028646.0,51,0,0.687194,31.328840,21.529004
79,2025-08-01,2350187.0,1510146.0,1673019.0,1527168.0,145851.0,77736256.0,51,0,0.711866,30.232830,21.521734
80,2025-09-01,2432300.0,1531376.0,1720750.0,1567903.0,152847.0,77486188.0,51,0,0.707458,31.390110,22.207184
81,2025-10-01,2412842.0,1468838.0,1719726.0,1566615.0,153111.0,77208947.0,51,0,0.712739,31.250808,22.273662
82,2025-11-01,2979207.0,1399095.0,1484804.0,1337807.0,146997.0,76473554.0,51,0,0.498389,38.957350,19.415915
83,2025-12-01,3062082.0,1654613.0,1915036.0,1741807.0,173229.0,76230910.0,51,0,0.625403,40.168509,25.121516


In [8]:
state_ops = state_eligibility_operations_summary(df)
state_ops.head(10)

,state_abbreviation,state_name,total_applications_submitted,medicaid_chip_agency_applications,total_medicaid_and_chip_determinations,medicaid_determinations,chip_determinations,average_monthly_enrollment,latest_month,determinations_per_application,applications_per_1000_average_enrollees,determinations_per_1000_average_enrollees
34,NY,New York,32246301.0,0.0,9097998.0,8675915.0,422083.0,6.760774e+06,2026-02-01,0.282141,4769.616689,1345.703592
9,FL,Florida,25060921.0,25060921.0,13900974.0,12393696.0,1507278.0,4.098383e+06,2026-02-01,0.554687,6114.831091,3391.819000
4,CA,California,14926149.0,11539297.0,13408368.0,12506434.0,901934.0,1.306298e+07,2026-02-01,0.898314,1142.630058,1026.440531
47,WA,Washington,12907381.0,6500893.0,4427855.0,4313662.0,114193.0,1.907782e+06,2026-02-01,0.343048,6765.649278,2320.944426
43,TX,Texas,9045451.0,9045451.0,5657393.0,4927465.0,729928.0,4.738800e+06,2026-02-01,0.625441,1908.806247,1193.845072
20,MD,Maryland,8929963.0,502246.0,5298327.0,4691082.0,607245.0,1.509452e+06,2026-02-01,0.593320,5916.030794,3510.100287
38,PA,Pennsylvania,8215717.0,7542700.0,7688274.0,7057317.0,630957.0,3.283032e+06,2026-02-01,0.935801,2502.478546,2341.821260
3,AZ,Arizona,6747708.0,6747708.0,921646.0,844710.0,76936.0,1.963490e+06,2026-02-01,0.136587,3436.588219,469.391649
35,OH,Ohio,5522349.0,5522349.0,3516157.0,3321514.0,194643.0,2.959433e+06,2026-02-01,0.636714,1866.015816,1188.118421
14,IL,Illinois,5245050.0,4449880.0,3195794.0,2906061.0,289733.0,3.347628e+06,2026-02-01,0.609297,1566.795843,954.644237


Short interpretation: Applications and determinations are available and suitable for an eligibility operations section. National applications peak in January 2024, while determinations peak in December 2023 in the current dashboard-ready table. These patterns should be framed as descriptive operational indicators, not as proof of administrative performance or policy impact.

## 5. Data Quality And Reporting Analysis

What did I do? Summarized missingness by field, state, and month; preliminary/final reporting; zero counts; and operational fields requiring caution.

Why did I do it? A healthcare reporting dashboard should show data quality context so users do not overinterpret incomplete fields.

What does it mean? Core enrollment, applications, and determinations are strong enough for dashboarding, while adult enrollment, call center, and processing-time fields require caveats.

Limitations: Missingness summaries identify reporting gaps but do not explain why a state or field is missing.

In [9]:
quality_field = data_quality_by_field(df)
quality_state = data_quality_by_state(df)
quality_month = data_quality_by_month(df)

print("Highest field missingness:")
display(quality_field.head(12))
print("Highest state missingness:")
display(quality_state.head(10))
print("Latest months reporting status:")
display(quality_month.tail(12))

Highest field missingness:


,field_name,missing_count,missing_percent,zero_count,zero_percent,use_caution
18,total_adult_medicaid_enrollment,3366,76.74,1,0.02,True
26,average_call_center_abandonment_rate,2590,59.05,2,0.05,True
24,total_call_center_volume_number_of_calls,2590,59.05,0,0.00,True
25,average_call_center_wait_time_minutes,2589,59.03,209,4.77,True
23,total_medicaid_and_chip_determinations_process...,1377,31.40,289,6.59,True
22,total_medicaid_and_chip_determinations_process...,1377,31.40,186,4.24,True
20,total_medicaid_and_chip_determinations_process...,1377,31.40,69,1.57,True
21,total_medicaid_and_chip_determinations_process...,1377,31.40,69,1.57,True
19,total_medicaid_and_chip_determinations_process...,1377,31.40,47,1.07,True
9,applications_for_financial_assistance_submitte...,0,0.00,3326,75.83,False


Highest state missingness:


,state_abbreviation,state_name,record_count,missing_value_count,missing_value_percent,preliminary_records
41,SD,South Dakota,86,459,20.53,1
39,RI,Rhode Island,86,357,15.97,1
3,AZ,Arizona,86,354,15.83,1
33,NV,Nevada,86,353,15.79,1
38,PA,Pennsylvania,86,351,15.70,1
28,ND,North Dakota,86,351,15.70,1
29,NE,Nebraska,86,351,15.70,1
30,NH,New Hampshire,86,351,15.70,1
31,NJ,New Jersey,86,351,15.70,1
32,NM,New Mexico,86,351,15.70,1


Latest months reporting status:


,reporting_month,state_records,missing_value_count,missing_value_percent,preliminary_records,all_records_final
74,2025-03-01,51,3,0.23,0,True
75,2025-04-01,51,3,0.23,0,True
76,2025-05-01,51,3,0.23,0,True
77,2025-06-01,51,3,0.23,0,True
78,2025-07-01,51,3,0.23,0,True
79,2025-08-01,51,3,0.23,0,True
80,2025-09-01,51,3,0.23,0,True
81,2025-10-01,51,3,0.23,0,True
82,2025-11-01,51,3,0.23,0,True
83,2025-12-01,51,3,0.23,0,True


Short interpretation: The latest month is preliminary for all states/DC. Adult enrollment, call center, and processing-time fields have the highest missingness and should not be headline KPIs unless clearly caveated. Zero values are present and should be checked during EDA before being interpreted as true operational performance.

## 6. Dashboard-Ready Tables

What did I do? Created clean CSV tables in `outputs/dashboard_tables/` for the future Plotly/Dash app.

Why did I do it? A dashboard should load simple, stable, tested tables rather than recomputing all EDA logic in the app.

What does it mean? The project is ready for app development after reviewing these tables and documentation.

Limitations: These tables are descriptive and should be revised if later EDA changes the dashboard scope.

In [10]:
tables = save_dashboard_tables(df)
save_figures(tables)
for name, table in tables.items():
    print(f"{name}: {len(table):,} rows")

kpi_summary: 8 rows
national_enrollment_trend: 86 rows
national_applications_determinations_trend: 86 rows
state_latest_snapshot: 51 rows
state_enrollment_change: 51 rows
state_eligibility_operations_summary: 51 rows
data_quality_by_field: 27 rows
data_quality_by_state: 51 rows
data_quality_by_month: 86 rows
dashboard_notes: 7 rows


In [11]:
tables['kpi_summary']

,metric,value,notes
0,latest_reporting_month,2026-02,Latest available reporting month in cleaned data.
1,latest_month_is_preliminary,True,Treat latest month cautiously when preliminary...
2,states_dc_included,51,All 50 states plus DC.
3,latest_total_medicaid_chip_enrollment,74877742,National sum across state/DC records.
4,monthly_enrollment_change,-918199,Change from prior reporting month.
5,change_since_january_2019,2711814,Descriptive change since first month in cleane...
6,latest_applications_submitted,2460804,National total applications for financial assi...
7,latest_total_determinations,1601892,National total Medicaid/CHIP eligibility deter...


## 7. Dashboard Notes

What did I do? Created plain-language notes for the eventual dashboard.

Why did I do it? Policy and operations users need clear caveats close to the metrics.

What does it mean? The future app should include notes about source, preliminary reporting, missingness, and unsupported claims.

Limitations: These notes are not a substitute for full source documentation.

In [12]:
dashboard_notes(df)

,note_type,note_text
0,data_source,Data source is the official CMS/Data.Medicaid....
1,latest_reporting_month_caution,The latest reporting month is 2026-02; 51 stat...
2,preliminary_final_reporting,CMS source data can include preliminary and up...
3,enrollment_interpretation,Enrollment fields are point-in-time month-end ...
4,application_determination_interpretation,Applications and determinations are descriptiv...
5,missingness_caveats,"Adult enrollment, call center, and processing-..."
6,dashboard_limits,"The dashboard cannot show individual outcomes,..."


## 8. Figures For README And Future Dashboard Design

The script creates four static figures in `outputs/figures/`:

- `national_enrollment_trend.png`
- `applications_determinations_trend.png`
- `top_state_enrollment_changes.png`
- `data_quality_missingness.png`

These are documentation and design aids, not the final interactive dashboard.

## 9. EDA Conclusion

The data is ready for Plotly/Dash app development after this EDA phase. The strongest first dashboard is a combined enrollment + eligibility operations dashboard with a visible data quality section. Forecasting should remain a later extension, and all interpretation should avoid causal claims.